# Expedia Hotel Recommendations


<img src='https://media.cntraveler.com/photos/685595770556b60be007dced/16:9/w_2864,h_1611,c_limit/062325-Best-Hotels-LA-W-Hollywood-PR_MG_2125-2-copy.jpg'>


Bu çalışmada `Expedia Hotel Recommendations` yarışması için kullanıcı arama ve rezervasyon bilgileri kullanılarak uygun otel gruplarını öneren bir başlangıç modeli oluşturulmuştur.


## Akış

1. Kütüphaneleri yükleme
2. Drive bağlama ve zip içeriğini tanımlama
3. Veri dosyalarını yükleme
4. Ön işleme
5. Son rezervasyon eğilimlerini çıkarma
6. Recommendation mantığı
7. Submission oluşturma
8. Sonuç


## 1. Kütüphaneleri Yükleme


In [1]:
# Bu bölümde otel grubu önerisi için gerekli veri hazırlama kütüphanelerini içe aktarıyoruz.


In [2]:
from google.colab import drive
import os
import zipfile
import tempfile

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)


## 2. Drive Bağlama ve Zip İçeriğini Tanımlama


In [3]:
# Bu bölümde yarışma zip dosyasını Drive üzerinden bağlıyor ve gerekli dosyaları buluyoruz.


In [ ]:
drive.mount('/content/drive')
zip_path = '/content/drive/MyDrive/Colab Data Dosyaları/expedia-hotel-recommendations.zip'
if not os.path.exists(zip_path):
    raise FileNotFoundError('Zip dosyası bulunamadı. Colab Data Dosyaları içindeki gerçek dosya adını kontrol et.')
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_members = zip_ref.namelist()
def find_zip_member(members, filename):
    for member in members:
        if member.endswith(filename):
            return member
    raise FileNotFoundError(f'{filename} zip içinde bulunamadi.')
train_member = find_zip_member(zip_members, 'train.csv')
test_member = find_zip_member(zip_members, 'test.csv')
destinations_member = find_zip_member(zip_members, 'destinations.csv')
sample_submission_member = find_zip_member(zip_members, 'sample_submission.csv')


## 3. Veri Dosyalarını Yükleme


In [6]:
sample_train_rows = 800000
sample_test_rows = 200000

def read_csv_from_zip(zip_path, member_name, nrows=None):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        with zip_ref.open(member_name) as f:
            return pd.read_csv(f, nrows=nrows)

train = read_csv_from_zip(zip_path, train_member, nrows=sample_train_rows)
test = read_csv_from_zip(zip_path, test_member, nrows=sample_test_rows)
destinations = read_csv_from_zip(zip_path, destinations_member)
sample_submission = read_csv_from_zip(zip_path, sample_submission_member)

print('Train shape (sample):', train.shape)
print('Test shape (sample):', test.shape)
print('Destinations shape:', destinations.shape)
print('Sample submission shape:', sample_submission.shape)
train.head()


Train shape (sample): (800000, 24)
Test shape (sample): (200000, 22)
Destinations shape: (62106, 150)
Sample submission shape: (2528243, 2)


,date_time,site_name,posa_continent,user_location_country,user_location_region,user_location_city,orig_destination_distance,user_id,is_mobile,is_package,channel,srch_ci,srch_co,srch_adults_cnt,srch_children_cnt,srch_rm_cnt,srch_destination_id,srch_destination_type_id,is_booking,cnt,hotel_continent,hotel_country,hotel_market,hotel_cluster
0,2014-08-11 07:46:59,2,3,66,348,48862,2234.2641,12,0,1,9,2014-08-27,2014-08-31,2,0,1,8250,1,0,3,2,50,628,1
1,2014-08-11 08:22:12,2,3,66,348,48862,2234.2641,12,0,1,9,2014-08-29,2014-09-02,2,0,1,8250,1,1,1,2,50,628,1
2,2014-08-11 08:24:33,2,3,66,348,48862,2234.2641,12,0,0,9,2014-08-29,2014-09-02,2,0,1,8250,1,0,1,2,50,628,1
3,2014-08-09 18:05:16,2,3,66,442,35390,913.1932,93,0,0,3,2014-11-23,2014-11-28,2,0,1,14984,1,0,1,2,50,1457,80
4,2014-08-09 18:08:18,2,3,66,442,35390,913.6259,93,0,0,3,2014-11-23,2014-11-28,2,0,1,14984,1,0,1,2,50,1457,21


## 4. Ön İşleme


In [7]:
# Bu bölümde tarih bilgilerini açıyor ve öneri için kullanacağımız bazı temel alanları hazırlıyoruz.


In [8]:
train_df = train.copy()
test_df = test.copy()

for df in [train_df, test_df]:
    df['date_time'] = pd.to_datetime(df['date_time'], errors='coerce')
    df['srch_ci'] = pd.to_datetime(df['srch_ci'], errors='coerce')
    df['srch_co'] = pd.to_datetime(df['srch_co'], errors='coerce')

    df['search_month'] = df['date_time'].dt.month
    df['search_year'] = df['date_time'].dt.year
    df['stay_length'] = (df['srch_co'] - df['srch_ci']).dt.days
    df['booking_month'] = df['srch_ci'].dt.month

print('Train null count:', train_df.isnull().sum().sum())
print('Test null count:', test_df.isnull().sum().sum())
train_df.head()


Train null count: 303715
Test null count: 65481


,date_time,site_name,posa_continent,user_location_country,user_location_region,user_location_city,orig_destination_distance,user_id,is_mobile,is_package,channel,srch_ci,srch_co,srch_adults_cnt,srch_children_cnt,srch_rm_cnt,srch_destination_id,srch_destination_type_id,is_booking,cnt,hotel_continent,hotel_country,hotel_market,hotel_cluster,search_month,search_year,stay_length,booking_month
0,2014-08-11 07:46:59,2,3,66,348,48862,2234.2641,12,0,1,9,2014-08-27,2014-08-31,2,0,1,8250,1,0,3,2,50,628,1,8,2014,4.0,8.0
1,2014-08-11 08:22:12,2,3,66,348,48862,2234.2641,12,0,1,9,2014-08-29,2014-09-02,2,0,1,8250,1,1,1,2,50,628,1,8,2014,4.0,8.0
2,2014-08-11 08:24:33,2,3,66,348,48862,2234.2641,12,0,0,9,2014-08-29,2014-09-02,2,0,1,8250,1,0,1,2,50,628,1,8,2014,4.0,8.0
3,2014-08-09 18:05:16,2,3,66,442,35390,913.1932,93,0,0,3,2014-11-23,2014-11-28,2,0,1,14984,1,0,1,2,50,1457,80,8,2014,5.0,11.0
4,2014-08-09 18:08:18,2,3,66,442,35390,913.6259,93,0,0,3,2014-11-23,2014-11-28,2,0,1,14984,1,0,1,2,50,1457,21,8,2014,5.0,11.0


## 5. Son Rezervasyon Eğilimlerini Çıkarma


In [9]:
# Bu bölümde kullanıcı davranışına daha yakın bir öneri üretmek için rezervasyon yapılan otel gruplarını özetliyoruz.


In [10]:
booking_df = train_df[train_df['is_booking'] == 1].copy()

popular_clusters = booking_df['hotel_cluster'].value_counts().head(20).index.tolist()

search_dest_cluster = (
    booking_df.groupby(['srch_destination_id', 'hotel_market'])['hotel_cluster']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
    .rename(columns={'hotel_cluster': 'top_cluster'})
)

print('En popüler otel grupları:', popular_clusters[:10])
print('Destination-cluster map shape:', search_dest_cluster.shape)
search_dest_cluster.head()


En popüler otel grupları: [91, 48, 42, 59, 28, 18, 82, 95, 16, 21]
Destination-cluster map shape: (8537, 3)


,srch_destination_id,hotel_market,top_cluster
0,4,246,82
1,8,416,16
2,11,824,94
3,14,1434,20
4,16,419,7


## 6. Recommendation Mantığı


In [11]:
# Bu bölümde test kullanıcıları için önce arama hedefi bazlı, yoksa genel popülerlik bazlı öneri üretiyoruz.


In [12]:
test_reco = test_df[['id', 'srch_destination_id', 'hotel_market']].merge(
    search_dest_cluster,
    on=['srch_destination_id', 'hotel_market'],
    how='left'
)

def build_recommendation(row):
    recs = []
    if pd.notna(row['top_cluster']):
        recs.append(str(int(row['top_cluster'])))
    recs.extend([str(x) for x in popular_clusters if str(x) not in recs])
    return ' '.join(recs[:5])

test_reco['hotel_cluster'] = test_reco.apply(build_recommendation, axis=1)

test_reco[['id', 'hotel_cluster']].head()


,id,hotel_cluster
0,0,55 91 48 42 59
1,1,91 48 42 59 28
2,2,0 91 48 42 59
3,3,1 91 48 42 59
4,4,48 91 42 59 28


## 7. Submission Oluşturma


In [13]:
# Bu bölümde submission dosyasını oluşturuyoruz.


In [14]:
submission = sample_submission[['id']].merge(
    test_reco[['id', 'hotel_cluster']],
    on='id',
    how='left'
)
submission['hotel_cluster'] = submission['hotel_cluster'].fillna(' '.join([str(x) for x in popular_clusters[:5]]))

print('Submission shape:', submission.shape)
submission.head()


Submission shape: (2528243, 2)


,id,hotel_cluster
0,0,55 91 48 42 59
1,1,91 48 42 59 28
2,2,0 91 48 42 59
3,3,1 91 48 42 59
4,4,48 91 42 59 28


In [15]:
output_path = '/content/expedia_submission.csv'
submission.to_csv(output_path, index=False)
print('Kaydedilen dosya:', output_path)


Kaydedilen dosya: /content/expedia_submission.csv


## 8. Sonuç


 Bu çalışmada Expedia Hotel Recommendations yarışması için kullanıcı arama ve rezervasyon bilgileri kullanılarak otel grubu önerisi yapan bir başlangıç modeli oluşturuldu. Elde edilen sonuçta rezervasyon davranışlarından çıkarılan eğilimler kullanılarak test kullanıcıları için otel grubu öneri listeleri üretildi.